# baidu/Unlimited-OCR trial (Colab, full quality)

Standalone notebook to try `baidu/Unlimited-OCR` with **full `crop_mode=True` tiling** — the setting that gives good quality on the model's HF Space demo, but OOMs on an 8GB laptop GPU. Colab's free T4 (16GB) or a paid A100 gives enough headroom to actually run it.

**Before running anything:** Runtime → Change runtime type → GPU (T4 is fine to start; A100 if you have Colab Pro and want more headroom for bigger pages / crop tiling).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Install dependencies

Colab already ships a CUDA-matched `torch` — don't reinstall it. The model's vendored code (`trust_remote_code=True`) needs `transformers==4.57.1` specifically; newer 5.x breaks it.

In [ ]:
!pip install -q "transformers==4.57.1" accelerate einops addict easydict matplotlib pymupdf pillow

## 2. Upload a document

Upload a PDF or image (e.g. one of your `data/raw_pdfs/*.pdf` files).

In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded.keys()))
print(f"Uploaded: {input_path}")

## 3. Helpers — render pages, Arabic-ratio quality signal

In [ ]:
import io, re, os
from pathlib import Path

_ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

def arabic_char_ratio(text: str) -> float:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    arabic = sum(1 for c in letters if _ARABIC_RE.match(c))
    return round(arabic / len(letters), 4)

def render_pages(path: str) -> list[bytes]:
    """PDF -> one full-color JPEG per page at dpi=200. Plain image -> passthrough, unchanged."""
    data = Path(path).read_bytes()
    if data[:4] != b"%PDF":
        return [data]
    import fitz
    from PIL import Image as PILImage
    doc = fitz.open(stream=data, filetype="pdf")
    pages = []
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img = PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=95)
        pages.append(buf.getvalue())
    doc.close()
    return pages

pages = render_pages(input_path)
print(f"{len(pages)} page(s) rendered")

## 4. Load the model

In [ ]:
import torch, time
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "baidu/Unlimited-OCR"

t0 = time.monotonic()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    dtype=torch.bfloat16,
).eval().cuda()
print(f"Model loaded in {time.monotonic() - t0:.1f}s")

## 5. Run inference — full quality (crop_mode tiling on)

This is the config that OOMs on an 8GB card. On a 16GB+ Colab GPU it should fit.

Notes from local debugging (see `scripts/try_unlimited_ocr.py` for the full story):
- `model.infer()` only **returns** the decoded text if `eval_mode=True` — otherwise it streams to stdout and returns `None`.
- `no_repeat_ngram_size=0` (the model's own default) lets it loop forever once uncertain — we set it to `3`.
- `base_size`/`crop_mode` here are the model's own defaults (`1024` / `True`) — the actual tiling that needs the extra VRAM.

In [ ]:
import tempfile

PROMPT = "<image>document parsing."
OUT_DIR = "/content/unlimited_ocr_out"
os.makedirs(OUT_DIR, exist_ok=True)

results = []
with tempfile.TemporaryDirectory() as tmpdir:
    for i, page_bytes in enumerate(pages):
        page_path = f"{tmpdir}/page_{i+1:03d}.jpg"
        Path(page_path).write_bytes(page_bytes)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.monotonic()
        with torch.inference_mode():
            text = model.infer(
                tokenizer,
                prompt=PROMPT,
                image_file=page_path,
                output_path=OUT_DIR,
                base_size=1024,
                image_size=640,
                crop_mode=True,
                max_length=32768,
                no_repeat_ngram_size=3,
                temperature=0.0,
                eval_mode=True,
            )
        torch.cuda.empty_cache()
        latency_ms = round((time.monotonic() - t0) * 1000, 1)
        peak_vram_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)

        ratio = arabic_char_ratio(text)
        print(f"\n--- page {i+1}/{len(pages)} — {latency_ms}ms, {len(text)} chars, "
              f"arabic_ratio={ratio}, peak_vram={peak_vram_gb}GB ---")
        print(text[:500])

        out_path = f"{OUT_DIR}/page_{i+1:03d}.txt"
        Path(out_path).write_text(text, encoding="utf-8")
        results.append({"page": i + 1, "latency_ms": latency_ms, "char_count": len(text),
                         "arabic_ratio": ratio, "peak_vram_gb": peak_vram_gb, "out_path": out_path})

results

## 6. (Optional) Download the output

In [ ]:
from google.colab import files
for r in results:
    files.download(r["out_path"])